In [ ]:
!pip install -q google-genai gradio

In [ ]:
from google import genai
from getpass import getpass

GEMINI_API_KEY = getpass("Enter your Gemini API Key: ")

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini AI Connected Successfully!")

Enter your Gemini API Key: ··········
Gemini AI Connected Successfully!


In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Suggest 3 creative names for a clothing business."
)

print(response.text)

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://aistudio.google.com/docs/get-started).', 'status': 'NOT_FOUND'}}

In [ ]:

try:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents="Suggest 3 creative names for a clothing business."
    )
    print(response.text)

except Exception as e:
    print("ERROR DETAILS:")
    print(e)


ERROR DETAILS:
404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://aistudio.google.com/docs/get-started).', 'status': 'NOT_FOUND'}}


In [ ]:
response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Suggest 3 creative names for a clothing business."
)

print(response.text)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:

try:
    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents="Suggest 3 creative names for a clothing business."
    )
    print("SUCCESS!")
    print(response.text)

except Exception as e:
    print("ERROR TYPE:", type(e).__name__)
    print("FULL ERROR:", repr(e))


ERROR TYPE: ServerError
FULL ERROR: ServerError("503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}")


In [ ]:

for model in client.models.list():
    if "flash" in model.name.lower():
        print(model.name)


models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-omni-1.2-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


In [ ]:

try:
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents="Suggest 3 creative names for a clothing business."
    )

    print("SUCCESS! Gemini AI Connected.")
    print(response.text)

except Exception as e:
    print("ERROR:", repr(e))


SUCCESS! Gemini AI Connected.
To give you the best suggestions, I have categorized these by "vibe." Here are three creative names for a clothing business:

### 1. The Modern/Minimalist Choice: **"Thread & Theory"**
*   **Why it works:** It sounds sophisticated, intellectual, and high-end. It implies that your clothing isn’t just fabric, but a well-thought-out design concept. It works perfectly for a brand that focuses on clean lines, capsule wardrobes, or sustainable fashion.

### 2. The Bold/Edgy Choice: **"Static Rebellion"**
*   **Why it works:** This name feels energetic and disruptive. It’s perfect for a streetwear brand, urban fashion, or a store that focuses on bold, expressive pieces. It suggests that your clothes are meant to make a statement and stand out in the "noise" of everyday life.

### 3. The Whimsical/Story-Driven Choice: **"Fable & Fold"**
*   **Why it works:** This creates a sense of nostalgia and craftsmanship. It sounds boutique-style, suggesting that every piece 

In [ ]:

!pip install -q gradio


In [ ]:

import gradio as gr
import json
import re

# Gemini AI Business Name Generator
def generate_business_ideas(business_idea, audience, style):

    if not business_idea.strip():
        return "Please enter your business idea.", gr.update(choices=[], value=[])

    if not audience.strip():
        return "Please enter your target audience.", gr.update(choices=[], value=[])

    prompt = f"""
    You are a professional branding expert.

    Business Idea: {business_idea}
    Target Audience: {audience}
    Branding Style: {style}

    Generate exactly 10 creative and unique business names.
    Give each name one catchy slogan.
    Also generate a business pitch of exactly 3 short lines.

    Return only valid JSON in this format:
    {{
      "ideas": [
        {{"name": "Business Name", "slogan": "Catchy slogan"}}
      ],
      "pitch": "Line 1\\nLine 2\\nLine 3"
    }}

    Make names memorable, easy to pronounce and relevant.
    """

    try:
        response = client.models.generate_content(
            model="gemini-3.1-flash-lite",
            contents=prompt,
            config={"response_mime_type": "application/json"}
        )

        text_response = response.text
        match = re.search(r"\{.*\}", text_response, re.DOTALL)

        if not match:
            raise ValueError("AI did not return valid JSON.")

        data = json.loads(match.group())
        ideas = data.get("ideas", [])[:10]
        pitch = data.get("pitch", "")

        if not ideas:
            raise ValueError("No business names generated.")

        result = "## ✨ Your AI-Generated Business Names\n\n"

        for i, item in enumerate(ideas, 1):
            name = item.get("name", "Unnamed")
            slogan = item.get("slogan", "No slogan")
            result += f"### {i}. {name}\n"
            result += f"**Slogan:** {slogan}\n\n"

        result += "## 🚀 Your Business Pitch\n\n"
        result += pitch

        names = [item.get("name", "") for item in ideas if item.get("name")]

        return result, gr.update(choices=names, value=[])

    except Exception as e:
        return f"Error: {str(e)}", gr.update(choices=[], value=[])

# Save favourite business names
def save_favourites(selected_names):

    if not selected_names:
        return "Please select at least one favourite name."

    result = "## ❤️ Your Favourite Business Names\n\n"

    for name in selected_names:
        result += f"- {name}\n"

    return result

# Gradio User Interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# ✨ Business Name & Slogan Generator")
    gr.Markdown(
        "### Create your brand identity using Generative AI!"
    )

    business_idea = gr.Textbox(
        label="Enter Your Business Idea",
        placeholder="Example: Clothing brand"
    )

    audience = gr.Textbox(
        label="Target Audience",
        placeholder="Example: College students"
    )

    style = gr.Dropdown(
        choices=[
            "Modern",
            "Professional",
            "Luxury",
            "Creative",
            "Traditional",
            "Trendy"
        ],
        value="Modern",
        label="Select Branding Style"
    )

    generate_btn = gr.Button(
        "✨ Generate Business Ideas",
        variant="primary"
    )

    output = gr.Markdown()

    gr.Markdown("### Select Your Favourite Names")

    favourites = gr.CheckboxGroup(
        choices=[],
        label="Choose your favourite business names"
    )

    save_btn = gr.Button("❤️ Save Favourite Names")

    favourite_output = gr.Markdown()

    generate_btn.click(
        fn=generate_business_ideas,
        inputs=[business_idea, audience, style],
        outputs=[output, favourites]
    )

    save_btn.click(
        fn=save_favourites,
        inputs=[favourites],
        outputs=[favourite_output]
    )

demo.launch(share=True)

/tmp/ipykernel_2148/1484236502.py:88: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4270aff79964d5b94a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
